# Work for Final Project Notebook

## Import Data

In [70]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots

## Clean & Set Up Data

In [2]:
# import .tsv as .csv
df = pd.read_csv('2025_specimen_time_series_events_no_phi.tsv', sep='\t')

In [3]:
df.head()

,accession_id,pat_enc_csn_id,pat_mrn_id,barcode,tube_id,specimen_id,test_code,test_performing_dept,test_performing_location,event_street,event_source,event_type,event_dt
0,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
1,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
2,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,FT4,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
3,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z
4,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z


Reshape to long format (one row per specimen-event) - this way, we can easily compute durations to each event type & compare across specimens

In [ ]:
# make sure event_dt is datetime
df['event_dt'] = pd.to_datetime(df['event_dt'])

# pivot to get one column per event type
# if there are multiple events of the same type for a specimen, we take 
# the earliest one (e.g. earliest resulted_dt)
df['test_id'] = df['accession_id'].astype(str) + "_" + df['test_code']

# only use order events to define test_id and timeline, since other events may be missing or duplicated
order_df = df[df['event_source'] == 'order'].copy()

timeline = order_df.pivot_table(
    index='test_id',
    columns='event_type',
    values='event_dt',
    aggfunc='min'  # in case of duplicates, take earliest
).reset_index()

# rename columns for clarity
# prefix with 'test_' to indicate these are test-related events
timeline.columns.name = None
timeline = timeline.rename(columns={
    'collected_dt': 'test_collected_dt',
    'receipt_dt': 'test_receipt_dt',
    'resulted_dt': 'test_min_resulted_dt',
    'verified_dt': 'test_min_verified_dt',
    'cancelled_dt': 'test_cancelled_dt',
    'cancellation_dt': 'test_cancelled_dt'
})

In [ ]:
# define start time as earliest available event for each specimen
# use as a reference point to compute durations to other events
event_cols = [
    'test_collected_dt', 'test_receipt_dt',
    'test_min_resulted_dt', 'test_min_verified_dt',
    'test_cancelled_dt'
]
# only consider event columns that actually exist in the data
existing_event_cols = [col for col in event_cols if col in timeline.columns]

# compute start time as min of existing event columns
timeline['start'] = timeline['test_ordered_dt']

# now compute durations in hours
# use only existing event columns to avoid errors
for col in existing_event_cols:
    timeline[col + '_hrs'] = (timeline[col] - timeline['start']).dt.total_seconds() / 3600

In [18]:
timeline.describe()

,test_collected_dt_hrs,test_receipt_dt_hrs,test_min_resulted_dt_hrs,test_min_verified_dt_hrs,test_cancelled_dt_hrs
count,229447.000000,224505.000000,219341.000000,217589.000000,9950.000000
mean,-1.662869,1.427063,3.890947,6.663868,7.796382
std,137.200319,138.256973,38.795072,41.948895,21.059802
min,-45860.666667,-45860.666667,-240.000000,-176.650000,-45.216667
25%,0.000000,0.233333,0.350000,0.550000,0.600000
50%,0.100000,0.716667,1.250000,1.400000,1.750000
75%,0.466667,2.666667,3.983333,4.433333,5.100000
max,354.733333,642.700000,15567.900000,15591.883333,703.433333


In [50]:
# remove rows where collected happens before ordered (bad data? assumption)
timeline = timeline[
    (timeline['test_collected_dt'].isna()) |
    (timeline['test_collected_dt'] >= timeline['test_ordered_dt'])
]

In [51]:
timeline.describe() 

,test_collected_dt_hrs,test_receipt_dt_hrs,test_min_resulted_dt_hrs,test_min_verified_dt_hrs,test_cancelled_dt_hrs,order_day
count,182076.000000,177518.000000,173411.000000,173242.000000,8702.000000,182982.000000
mean,0.658631,2.982987,4.913850,5.491968,7.862082,2.521926
std,2.281814,6.594367,41.266058,42.220810,20.917295,1.816473
min,0.000000,-0.016667,-16.516667,-0.316667,-2.066667,0.000000
25%,0.050000,0.516667,0.850000,0.966667,0.716667,1.000000
50%,0.200000,1.033333,1.616667,1.733333,1.900000,2.000000
75%,0.616667,3.600000,4.800000,4.933333,5.350000,4.000000
max,354.733333,642.700000,15567.900000,15591.883333,703.433333,6.000000


## Visualizations

### Visualization 1
Visualize (visual, inspectable, graphic timelines) the “average” time series of events for laboratory orders, given a set of dimensional filters (such as “all CBCs ordered in ward CW1”).  Method is open to interpretation, and whichever means the students find most readily available.

In [53]:
# get the duration-related columns
duration_cols = [col for col in timeline.columns if col.endswith('_hrs')]

# make it into long format
long_df = timeline.melt(
    id_vars=['test_id'],  #just filter by test_id for simplicity now
    value_vars=duration_cols, # only melt the duration columns
    var_name='event_stage', # this will be the x-axis in the dashboard
    value_name='hours' #y-axis
)

# clean the labels, just in case
long_df['event_stage'] = (
    long_df['event_stage']
    .str.replace('_hrs', '')
    .str.replace('test_', '')
)

In [30]:
# timeline currently has test_id but not test_code, so derive it safely
if 'test_code' not in timeline.columns:
    timeline = timeline.copy()
    timeline['test_code'] = timeline['test_id'].str.rsplit('_', n=1).str[-1]

long_df = timeline.melt(
    id_vars=['test_id', 'test_code'],  # add ward if available
    value_vars=duration_cols,
    var_name='event_stage',
    value_name='hours'
)

In [54]:
# make the dashboard function that can be called with different filters (e.g. test_code)
# create the interactive timeline (x-axis: event stage, y-axis: hours from start)

def plot_interactive_timeline(data, title="Average Timeline"):
    
    # compute average hours for each event stage
    avg_df = (
        data.groupby('event_stage')['hours']
        .mean()
        .reset_index()
    )
    
    # create line plot with markers for each event stage
    fig = px.line(
        avg_df,
        x='event_stage',
        y='hours',
        markers=True,
        title=title
    )
    
    # add hover info to show exact hours for each stage
    fig.update_layout(
        xaxis_title="Event Stage",
        yaxis_title="Hours from Start",
        hovermode="x unified"
    )
    
    # show the figure
    fig.show()

In [57]:
# filter on test_code (example)

# get the unique test codes for the dropdown options, including an "All" option
test_options = ['All'] + sorted(timeline['test_code'].dropna().unique().tolist())

# make the dropdown widget for test_code selection
dropdown = widgets.Dropdown(
    options=test_options,
    description='Test Code:',
    value='All'
)

In [58]:
# create the interactive dashboard that updates based on the selected test_code 
def update_dashboard(test_code):
    
    # if "All" is selected, use the full dataset; otherwise filter by the selected test_code
    if test_code == 'All':
        data = long_df.copy()
    else:
        data = long_df[long_df['test_code'] == test_code]
    
    # compute average hours for each event stage
    avg_df = (
        data.groupby('event_stage')['hours'] # group by event stage and compute mean hours
        .mean()
        .reset_index()
    )
    # create line plot with markers for each event stage
    fig = px.line(
        avg_df,
        x='event_stage',
        y='hours',
        markers=True,
        title=f"Average Timeline ({test_code})"
    )
    
    # update layout to show info for each stage when hovered
    fig.update_layout(
        hovermode="x unified"
    )
    
    fig.show()

# display the dropdown and link it to the update function
widgets.interactive(update_dashboard, test_code=dropdown)

interactive(children=(Dropdown(description='Test Code:', options=('All', '17HP2', '21OH', '25HD', '6MMP', 'A1A…

### Visualization 2

An A/B version of the above visualization to compare two mutually exclusive datasets (such as “All CBCs ordered in ward CW1, tests ordered on weekdays vs tests ordered on weekends”).

In [59]:
# first, create weekday/weekend groups based on order day
timeline['order_day'] = timeline['start'].dt.dayofweek

# define group as "Weekend" if order_day is 5 or 6, otherwise "Weekday"
timeline['group'] = timeline['order_day'].apply(
    lambda x: 'Weekend' if x >= 5 else 'Weekday'
)

In [60]:
# prep data for plotly
# get the duration-related columns again for clarity
duration_cols = [col for col in timeline.columns if col.endswith('_hrs')]

# make it into long format again, now including the group variable for coloring
long_df = timeline.melt(
    id_vars=['test_id', 'group'], #use group for coloring
    value_vars=duration_cols,
    var_name='event_stage',
    value_name='hours'
)

# clean labels just in case
long_df['event_stage'] = (
    long_df['event_stage']
    .str.replace('_hrs', '')
    .str.replace('test_', '')
)

In [61]:
# make the A/B comparison plot for weekday vs weekend specimen journey
import plotly.express as px

# compute average hours for each event stage and group
ab_df = (
    long_df.groupby(['group', 'event_stage'])['hours']
    .mean()
    .reset_index()
)

# create line plot with markers for each event stage, colored by group
fig = px.line(
    ab_df,
    x='event_stage',
    y='hours',
    color='group',
    markers=True,
    title="A/B Comparison: Weekday vs Weekend Specimen Journey"
)

# update layout to show info for each stage when hovered
fig.update_layout(
    xaxis_title="Event Stage",
    yaxis_title="Hours from Order",
    hovermode="x unified"
)

# show the figure
fig.show()

In [62]:
# make interactive with test_code filter
# get the unique test codes for the dropdown options, including an "All" option
test_options = ['All'] + sorted(timeline['test_code'].dropna().unique())

# create the dropdown widget for test_code selection
dropdown = widgets.Dropdown(
    options=test_options,
    description='Test Code:',
    value='All'
)

In [63]:
# create the interactive dashboard that updates based on the selected test_code, now with A/B comparison
def update_ab_dashboard(test_code):
    
    # if "All" is selected, use the full dataset; otherwise filter by the selected test_code
    if test_code == 'All':
        data = timeline.copy()
    else:
        data = timeline[timeline['test_code'] == test_code]
    
    # melt (put into long format again) after filtering
    long_df = data.melt(
        id_vars=['test_id', 'group'],
        value_vars=duration_cols,
        var_name='event_stage',
        value_name='hours'
    )
    
    # clean the labels again just in case
    long_df['event_stage'] = (
        long_df['event_stage']
        .str.replace('_hrs', '')
        .str.replace('test_', '')
    )
    
    # compute average hours for each event stage and group after filtering
    ab_df = (
        long_df.groupby(['group', 'event_stage'])['hours']
        .mean()
        .reset_index()
    )
    
    # create line plot with markers for each event stage, colored by group
    fig = px.line(
        ab_df,
        x='event_stage',
        y='hours',
        color='group',
        markers=True,
        title=f"A/B Timeline ({test_code})"
    )
    
    # update layout to show info for each stage when hovered
    fig.update_layout(hovermode="x unified")
    fig.show()

# display the dropdown and link it to the update function
widgets.interactive(update_ab_dashboard, test_code=dropdown)

interactive(children=(Dropdown(description='Test Code:', options=('All', '17HP2', '21OH', '25HD', '6MMP', 'A1A…

### Visualization 3

Some way to visualize the likelihood of a named time series event, such as “specimen cancelled due to hemolysis”, in that A/B comparison.

In [ ]:
# get our defined metrics of interest (ex. cancellation)
timeline['cancelled'] = timeline['test_cancelled_dt'].notna()

# another - delayed event (> 4hrs, assumption)
timeline['delayed_result'] = timeline['test_min_resulted_dt_hrs'] > 4

In [64]:
# calculate probabilities of cancellation and delay for each group
prob_df = timeline.groupby('group').agg({
    'cancelled': 'mean',
    'delayed_result': 'mean'
}).reset_index()

# melt to long format for easier plotting
prob_df = prob_df.melt(
    id_vars='group',
    var_name='event_type',
    value_name='probability'
)

In [66]:
# make our interactive comparison plot for event likelihoods (cancellation and delay) by group

# create bar plot comparing probabilities of cancellation and delay between groups
fig = px.bar(
    prob_df,
    x='event_type',
    y='probability',
    color='group',
    barmode='group',
    text_auto=True,
    title="Event Likelihood Comparison (A/B)"
)

# update layout to show info for each stage when hovered
fig.update_layout(
    yaxis_title="Probability",
    xaxis_title="Event Type"
)

# show the figure
fig.show()

In [67]:
# make the event likelihood comparison into a function that can be called with different filters (e.g. test_code)
def plot_event_likelihood(data):
    
    # calculate probabilities of cancellation and delay for each group
    prob_df = data.groupby('group').agg({
        'cancelled': 'mean',
        'delayed_result': 'mean'
    }).reset_index()
    
    # melt to long format for easier plotting
    prob_df = prob_df.melt(
        id_vars='group',
        var_name='event_type',
        value_name='probability'
    )
    
    # create bar plot comparing probabilities of cancellation and delay between groups
    fig = px.bar(
        prob_df,
        x='event_type',
        y='probability',
        color='group',
        barmode='group',
        text_auto=True,
        title="Event Likelihood Comparison"
    )
    
    # update layout to show info for each stage when hovered
    fig.show()

In [69]:
# connect it to our existing dropdown
def update_full_dashboard(test_code):
    
    # if "All" is selected, use the full dataset; otherwise filter by the selected test_code
    if test_code == 'All':
        data = timeline.copy()
    else:
        data = timeline[timeline['test_code'] == test_code]
    
    # ---- A/B TIMELINE ----
    long_df = data.melt(
        id_vars=['test_id', 'group'],
        value_vars=duration_cols,
        var_name='event_stage',
        value_name='hours'
    )
    
    # clean the labels again just in case
    long_df['event_stage'] = (
        long_df['event_stage']
        .str.replace('_hrs', '')
        .str.replace('test_', '')
    )
    
    # compute average hours for each event stage and group after filtering
    ab_df = (
        long_df.groupby(['group', 'event_stage'])['hours']
        .mean()
        .reset_index()
    )
    
    # create line plot with markers for each event stage, colored by group
    fig1 = px.line(
        ab_df,
        x='event_stage',
        y='hours',
        color='group',
        markers=True,
        title=f"A/B Timeline ({test_code})"
    )
    
    # show fig 1
    fig1.show()
    
    # ---- EVENT LIKELIHOOD ----
    plot_event_likelihood(data)

## Render Dashboard All Together

In [71]:
def render_dashboard(test_code='All'):
    
    # ---- FILTER ----
    # if "All" is selected, use the full dataset; otherwise filter by the selected test_code
    if test_code == 'All':
        data = timeline.copy()
    else:
        data = timeline[timeline['test_code'] == test_code]
    
    # ---- TIMELINE PREP ----
    # melt (put into long format again) after filtering, now including the group variable for coloring
    long_df = data.melt(
        id_vars=['test_id', 'group'],
        value_vars=duration_cols,
        var_name='event_stage',
        value_name='hours'
    )
    
    # clean the labels again just in case
    long_df['event_stage'] = (
        long_df['event_stage']
        .str.replace('_hrs', '')
        .str.replace('test_', '')
    )
    
    # compute average hours for each event stage and group after filtering
    ab_df = (
        long_df.groupby(['group', 'event_stage'])['hours']
        .mean()
        .reset_index()
    )
    
    # ---- TIMELINE FIG ----
    # create line plot with markers for each event stage, colored by group
    fig_timeline = px.line(
        ab_df,
        x='event_stage',
        y='hours',
        color='group',
        markers=True,
        title=f"A/B Timeline ({test_code})"
    )
    
    # update layout to show info for each stage when hovered
    fig_timeline.update_layout(
        hovermode="x unified",
        xaxis_title="Event Stage",
        yaxis_title="Hours from Order"
    )
    
    # ---- EVENT LIKELIHOOD PREP ----
    # find the probabilities of cancellation and delay for each group
    prob_df = data.groupby('group').agg({
        'cancelled': 'mean',
        'delayed_result': 'mean'
    }).reset_index()
    
    # melt to long format for easier plotting
    prob_df = prob_df.melt(
        id_vars='group',
        var_name='event_type',
        value_name='probability'
    )
    
    # ---- EVENT LIKELIHOOD FIG ----
    # create bar plot comparing probabilities of cancellation and delay between groups
    fig_prob = px.bar(
        prob_df,
        x='event_type',
        y='probability',
        color='group',
        barmode='group',
        text_auto=True,
        title="Event Likelihood"
    )
    
    # update layout to show info for each stage when hovered
    fig_prob.update_layout(
        yaxis_title="Probability"
    )
    
    # ---- COMBINED LAYOUT ----
    # combine the timeline and event likelihood figures into a single dashboard layout
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("A/B Timeline", "Event Likelihood")
    )
    
    # add timeline traces (lines and markers)
    for trace in fig_timeline.data:
        fig.add_trace(trace, row=1, col=1)
    
    # add probability traces (bars)
    for trace in fig_prob.data:
        fig.add_trace(trace, row=1, col=2)
    
    # update overall layout
    fig.update_layout(
        height=500,
        width=1000,
        title_text=f"Specimen Journey Dashboard ({test_code})"
    )
    
    # show the combined dashboard 
    fig.show()

In [74]:
# make interactive controls
# get the unique test codes for the dropdown options, including an "All" option
test_options = ['All'] + sorted(timeline['test_code'].dropna().unique())

# create the dropdown widget for test_code selection
dropdown = widgets.Dropdown(
    options=test_options,
    description='Test Code:',
    value='All'
)

# show the interactive dashboard that updates based on the 
# selected test_code, now with both A/B timeline and event likelihood comparison
widgets.interactive(render_dashboard, test_code=dropdown)

interactive(children=(Dropdown(description='Test Code:', options=('All', '17HP2', '21OH', '25HD', '6MMP', 'A1A…